# Create Nuffield Foundation Awards (GRANT PATTERN, REST + landing-page hybrid)

Nuffield Foundation is a major UK funder of social science / welfare /
education research (~£15M/yr, established 1943). Their project corpus
is exposed via the WordPress REST API at
`https://www.nuffieldfoundation.org/wp-json/wp/v2/project` (663
projects as of ingest), but the fields the awards table needs
(`amount` in GBP, date range, theme, lead researchers) live ONLY on
the per-project landing pages inside `<div class="article-meta"><svg
class="icon awarded">...£27,692</div>`-style blocks. The source
script does a **hybrid** ingest: REST for the project list +
landing-page scrape for the meta fields.

This is a new precedent in the awards pipeline. Existing scripts are
either REST-only (Holberg, CIFAR) or pure HTML scrape (Wolf, Lasker,
Fields, etc.). Future foundations whose REST API exposes only
navigation metadata can model on `nuffield_to_s3.py`.

**Awarding body:** Nuffield Foundation — F4320319997 (GB)

**Schema choices:**
- One row per project (no multi-prize splits — each Nuffield project
  is a single grant to one team).
- `funder_scheme` = the theme label from the landing page's icon
  metadata (e.g., `'Education'`, `'Justice'`, `'Welfare'`).
- `amount` populated from landing-page regex anchored on the `icon
  awarded` SVG marker; `currency = 'GBP'` hardcoded (single-country
  funder).
- `start_date` / `end_date` parsed from the `icon date` block's
  "Month Year - Month Year" range; both are first-of-month dates.
- `lead_investigator` is the first `<strong>` in the
  `Researchers: ...` block; remaining researchers move to
  `co_lead_investigator` (next) and `investigators` (rest).
- `description` is the cleaned `content.rendered` from the REST API
  (rich — typically 1-3 paragraphs of project summary).
- `declined` boolean populated for schema parity (always False).

**Prerequisites:** Run `scripts/local/nuffield_to_s3.py` first to
fetch from `nuffieldfoundation.org` + scrape landings, then upload
parquet to S3.

**Data source:** https://www.nuffieldfoundation.org/wp-json/wp/v2/project
**S3 location:** `s3a://openalex-ingest/awards/nuffield/nuffield_projects.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nuffield_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/nuffield/nuffield_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.nuffield_raw;

## Step 1.5: Inspect raw + money/currency scan

Per runbook §1.5, we explicitly scan for money-flavored columns even
though we already know the script parsed amounts from the landing
page. This catches drift if the source landing-page markup changes
out from under us.

Known source columns (all STRING per runbook §1.2.5):
`funder_award_id`, `wp_id`, `slug`, `title`, `description`, `theme`,
`amount`, `currency`, `start_date`, `end_date`, `lead_full_name`,
`lead_given_name`, `lead_family_name`, `co_investigators`,
`landing_page_url`, `first_seen_date`, `declined`.

Amounts: GBP, parsed from landing-page `icon awarded` markers. Expect
~95-100% coverage — a few legacy projects may lack the awarded amount
in the published markup.

In [ ]:
%sql
DESCRIBE openalex.awards.nuffield_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.nuffield_raw LIMIT 5;

In [ ]:
%sql
-- Money-shape scan per runbook §1.5. amount column already exists;
-- this confirms its distribution is in the expected GBP grant range.
SELECT
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(amount AS DOUBLE)) AS avg_amount,
    COUNT(amount) AS non_null,
    COUNT(*) AS total
FROM openalex.awards.nuffield_raw;

In [ ]:
%sql
-- Currency values present (should be only 'GBP' or NULL).
SELECT currency, COUNT(*) AS rows
FROM openalex.awards.nuffield_raw
GROUP BY currency;

## Step 1.6: Fail-fast — verify Nuffield funder row exists

The Step 2 transform `CROSS JOIN`s against `openalex.common.funder`.
If the funder row is missing, the join silently emits zero rows.
Assert presence here.

In [ ]:
%sql
-- Must return exactly 1 row. If 0, STOP — the funder is missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320319997;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nuffield_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320319997  -- Nuffield Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.title AS display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,  -- 'GBP' from script, NULL when amount missing
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    -- Nuffield funds research grants only (no fellowship/prize categories).
    'research' AS funding_type,
    n.theme AS funder_scheme,
    'nuffield_wp_rest' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date, 'yyyy-MM-dd') AS end_date,
    TRY_CAST(SUBSTRING(n.start_date, 1, 4) AS INT) AS start_year,
    TRY_CAST(SUBSTRING(n.end_date, 1, 4) AS INT) AS end_year,
    CASE
        WHEN n.lead_full_name IS NULL OR n.lead_full_name = '' THEN NULL
        ELSE struct(
            n.lead_given_name AS given_name,
            n.lead_family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            CAST(NULL AS STRUCT<name:STRING, country:STRING,
                ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>) AS affiliation
        )
    END AS lead_investigator,
    -- Second researcher (if any) lands in co_lead.
    CASE
        WHEN n.co_investigators IS NULL OR n.co_investigators = '' THEN NULL
        ELSE struct(
            CAST(NULL AS STRING) AS given_name,
            try_element_at(split(n.co_investigators, '\\|'), 1) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            CAST(NULL AS STRUCT<name:STRING, country:STRING,
                ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>) AS affiliation
        )
    END AS co_lead_investigator,
    -- Third onwards go to the investigators array (rare).
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.nuffield_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.title IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 80

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nuffield_wp_rest' AND priority = 80;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    80 AS priority  -- Nuffield priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.nuffield_awards;

## Step 6: Verification

Full §6.1-6.7 verification. Nuffield amount-coverage check is NOT
waived — expect ~95-100% `pct_amount`, single currency 'GBP'.

In [ ]:
%sql
SELECT COUNT(*) AS total_nuffield_award_rows FROM openalex.awards.nuffield_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.nuffield_awards;

In [ ]:
%sql
-- §6.3 Data completeness (runbook canonical query)
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.nuffield_awards;

In [ ]:
%sql
-- §6.7 amount + currency fail-fast check (NOT waived for Nuffield).
-- Expect: pct_amount >= 95, currencies = ['GBP'], avg in 10K-1M GBP.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount
FROM openalex.awards.nuffield_awards;

In [ ]:
%sql
-- Sample inspection
SELECT id, SUBSTRING(display_name, 1, 70) AS title,
       funder_scheme AS theme, amount, currency, start_year, end_year,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family
FROM openalex.awards.nuffield_awards
ORDER BY start_year DESC NULLS LAST, amount DESC NULLS LAST
LIMIT 12;

In [ ]:
%sql
-- Theme distribution — expect a handful of distinct themes (Education, Justice, Welfare, etc.).
SELECT funder_scheme AS theme, COUNT(*) AS rows, ROUND(SUM(amount)/1e6, 2) AS total_gbp_millions
FROM openalex.awards.nuffield_awards
GROUP BY funder_scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- Year distribution — most projects 2014-present (older projects may be missing in current site).
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.nuffield_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 20;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.nuffield_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;